In [0]:
# Faster search space specifically for large datasets like HIGGS
SEARCH_SPACE_FAST = {
    "lr": {
        "maxIter":         [10, 20, 50],        # no 100, 150, 200
        "regParam":        [0.0, 0.01, 0.1],
        "elasticNetParam": [0.0, 0.5, 1.0]
    },
    "rf": {
        "numTrees":            [10, 20, 50],     # no 100, 150, 200
        "maxDepth":            [3, 5, 8],        # no 10, 12, 15
        "minInstancesPerNode": [1, 4, 16],
        "subsamplingRate":     [0.6, 0.8, 1.0]
    },
    "gbt": {
        "maxIter":         [10, 20],             # GBT is slowest — keep very small
        "maxDepth":        [3, 4, 5],
        "stepSize":        [0.05, 0.1, 0.2],
        "subsamplingRate": [0.6, 0.8, 1.0]
    },
    "svm": {
        "maxIter":         [10, 20, 50],
        "regParam":        [0.001, 0.01, 0.1],
        "standardization": [True, False]
    }
}


def create_chromosome_fast():
    """Use smaller search space for large datasets."""
    model_type = random.choice(list(SEARCH_SPACE_FAST.keys()))
    params = {
        k: random.choice(v)
        for k, v in SEARCH_SPACE_FAST[model_type].items()
    }
    return (model_type, params)


def run_ga_fast(eval_fn, generations=3, pop_size=4):
    """GA using fast search space — for large datasets."""
    population = [create_chromosome_fast() for _ in range(pop_size)]
    history = []
    avg_history = []

    for gen in range(generations):
        print(f"\n=== Generation {gen} ===")
        scores = eval_fn(population)
        scores.sort(reverse=True, key=lambda x: x[0])

        best_score = scores[0][0]
        avg_score  = sum(s[0] for s in scores) / len(scores)
        history.append(best_score)
        avg_history.append(avg_score)

        print(f"Best fitness : {best_score:.4f}")
        print(f"Avg  fitness : {avg_score:.4f}")

        best_individual = scores[0][1]
        selected = [s[1] for s in scores[:3]]

        new_population = [best_individual]
        while len(new_population) < pop_size:
            p1, p2 = random.sample(selected, 2)

            # crossover within fast space
            model = p1[0]
            params = {}
            for key in p1[1]:
                params[key] = random.choice([
                    p1[1][key],
                    p2[1].get(key, p1[1][key])
                ])
            child = (model, params)

            # mutate within fast space
            if random.random() < 0.3:
                m, p = child
                for k in p:
                    if random.random() < 0.5:
                        p[k] = random.choice(SEARCH_SPACE_FAST[m][k])
                child = (m, p)

            new_population.append(child)

        population = new_population

    return population, history, avg_history